# RSNA Knee — five-fold EMA fused model submission

Raptor v5 CoAtNet + ImageNet ResNet-50 residual fusion, **five-fold EMA probability ensemble, no TTA**.
Weights: fold-selected EMA epochs 3, 3, 4, 3, and 1 from the minimum-overlap 384 V2 five-fold run.
Selections use only held-out-fold validation AUROC, never Gold74.
Test DICOMs are preprocessed with the exact minimum-overlap 384 V2 training recipe.

Embedded source digests: `{"models/__init__.py": "360790aae4faf61c", "models/raptor_v5_resnet50.py": "b9cb773923e4bb1a", "preprocess_six_slot_raptor.py": "cc5e92682fd0b691", "preprocess_series_safe_384.py": "557601b2bc903397", "preprocess_min_overlap_384.py": "7dd62b9d3241825e", "infer_cv5_submission.py": "dd013ba278331d3a"}`


In [ ]:
from pathlib import Path
import sys
SOURCES = {'models/__init__.py': '"""Trainable models for RSNA knee experiments."""\n', 'models/raptor_v5_resnet50.py': '"""Raptor v5 with residual ResNet50 fusion inside CoAtNet.\n\nInput is an already preprocessed [B, K, 3, 384, 384] triplet bag. Both\nencoders receive exactly the same tensor; preprocessing belongs to the loader.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.utils.checkpoint import checkpoint\n\n\nCOATNET_ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"\n# v3: bottleneck fusion (separate LNs, 1x1 down, DW 3x3, GELU, 1x1 up) with a\n# channel-wise gate on the residual. v1 (3-layer concatenation MLP) and v2\n# (ungated bottleneck) fused checkpoints do not load here.\nFUSED_FORMAT = "raptor_v5_resnet50_v3"\nLABELS = (\n    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s",\n    "Contusion", "Fracture",\n)\n\n\n@dataclass(frozen=True)\nclass FusionConfig:\n    stage2_hidden: int = 128\n    stage4_hidden: int = 256\n    # Hidden width of each channel-gate MLP; 0 disables gating (g = 1).\n    stage2_gate_hidden: int = 64\n    stage4_gate_hidden: int = 128\n    mlp_dropout: float = 0.0\n    head_dropout: float = 0.2\n    gradient_checkpointing: bool = False\n\n\nclass BottleneckFusion(nn.Module):\n    """Mix-FFN style residual fusion of two aligned feature maps.\n\n    LN_a(a), LN_b(b) -> concat -> 1x1 down (to `hidden`) -> depthwise 3x3\n    -> GELU -> Dropout -> 1x1 up (zero-initialized) = F, returned as g * F.\n\n    Each input is normalized separately, so neither backbone\'s feature scale\n    dominates the other\'s. The depthwise 3x3 lets every output position use\n    the fused features of its 3x3 neighbourhood, tolerating the slight\n    misalignment between the two trunks\' receptive fields. Only the up\n    projection starts at zero, so the residual is exactly zero initially\n    without a dead block.\n\n    Channel gate (gate_hidden > 0): g = sigmoid(W2 GELU(W1 p)), where p is\n    the spatial mean of the normalized, concatenated inputs. One gate value\n    per sample and output channel, shared by all positions, so each window\n    decides how much of each channel\'s correction to admit. The gate cannot\n    disturb the zero start because it multiplies a residual that is zero.\n    """\n\n    def __init__(self, first_channels: int, second_channels: int,\n                 out_channels: int, hidden: int, dropout: float = 0.0,\n                 gate_hidden: int = 0):\n        super().__init__()\n        if min(first_channels, second_channels, out_channels, hidden) <= 0:\n            raise ValueError("Fusion channel dimensions must be positive")\n        if gate_hidden < 0:\n            raise ValueError("gate_hidden must be nonnegative (0 disables the gate)")\n        self.first_channels = first_channels\n        self.second_channels = second_channels\n        self.norm_first = nn.LayerNorm(first_channels)\n        self.norm_second = nn.LayerNorm(second_channels)\n        self.down = nn.Linear(first_channels + second_channels, hidden)\n        self.dw = nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden)\n        self.act = nn.GELU()\n        self.drop = nn.Dropout(dropout)\n        self.up = nn.Conv2d(hidden, out_channels, 1)\n        nn.init.zeros_(self.up.weight)\n        nn.init.zeros_(self.up.bias)\n        # Default init with zero logit bias: gates start near 0.5 and are\n        # input-dependent from the first update on.\n        self.gate = (nn.Sequential(\n            nn.Linear(first_channels + second_channels, gate_hidden), nn.GELU(),\n            nn.Linear(gate_hidden, out_channels), nn.Sigmoid(),\n        ) if gate_hidden else None)\n        if self.gate is not None:\n            nn.init.zeros_(self.gate[2].bias)\n\n    def forward(self, first: Tensor, second: Tensor) -> Tensor:\n        if (first.ndim != 4 or second.ndim != 4\n                or first.shape[0] != second.shape[0]\n                or first.shape[2:] != second.shape[2:]\n                or first.shape[1] != self.first_channels\n                or second.shape[1] != self.second_channels):\n            raise ValueError(f"Incompatible fusion features: {first.shape}, {second.shape}")\n        a = self.norm_first(first.permute(0, 2, 3, 1))  # LayerNorm over channels\n        b = self.norm_second(second.permute(0, 2, 3, 1))\n        joined = torch.cat((a, b), dim=-1)  # [N, H, W, Ca+Cb]\n        h = self.down(joined).permute(0, 3, 1, 2)  # NCHW\n        residual = self.up(self.drop(self.act(self.dw(h))))\n        if self.gate is None:\n            return residual\n        g = self.gate(joined.mean(dim=(1, 2)))  # [N, Cout]: one gate per channel\n        return residual * g[:, :, None, None]\n\n\nclass ResNet50Encoder(nn.Module):\n    """ImageNet ResNet-50 trunk receiving the full shared three-slice triplet.\n\n    conv3_x is backbone[5] (torchvision layer2); conv5_x is backbone[7]\n    (layer4). A local checkpoint must use the full torchvision state layout,\n    including fc; it is strict-loaded before the classification head is removed.\n    """\n\n    def __init__(self, checkpoint_source: str | Path | Mapping[str, Any] | None = None,\n                 *, weights: str | None = "IMAGENET1K_V2"):\n        super().__init__()\n        from torchvision.models import ResNet50_Weights, resnet50\n\n        network = resnet50(weights=(ResNet50_Weights.verify(weights)\n                                   if checkpoint_source is None else None))\n        if checkpoint_source is not None:\n            ck = _checkpoint(checkpoint_source)\n            network.load_state_dict(ck.get("state_dict", ck), strict=True)\n        self.backbone = nn.Sequential(*list(network.children())[:-2])\n\n    def forward(self, image: Tensor) -> tuple[Tensor, Tensor]:\n        x = image\n        conv3 = None\n        for index, layer in enumerate(self.backbone):\n            x = layer(x)\n            if index == 5:\n                conv3 = x\n        return conv3, x\n\n\nclass FusedCoAtNet(nn.Module):\n    """Stage2 Z += g1*F1(X,Z), Stage4 V += g2*F2(V,Y), then normal CoAtNet pooling.\n\n    g1, g2 are the per-channel gates computed inside each fusion block.\n    """\n\n    def __init__(self, coatnet: nn.Module, resnet50: nn.Module,\n                 config: FusionConfig = FusionConfig()):\n        super().__init__()\n        self.coatnet = coatnet\n        self.resnet50 = resnet50\n        self.num_features = coatnet.num_features\n        channels = [entry["num_chs"] for entry in coatnet.feature_info]\n        if len(coatnet.stages) != 4 or channels[-4:] != [128, 256, 512, 1024]:\n            raise ValueError("Expected the four-stage Raptor v5 CoAtNet RMLP-2")\n        self.f1 = BottleneckFusion(512, 256, 256, config.stage2_hidden, config.mlp_dropout,\n                                   gate_hidden=config.stage2_gate_hidden)\n        self.f2 = BottleneckFusion(1024, 2048, 1024, config.stage4_hidden, config.mlp_dropout,\n                                   gate_hidden=config.stage4_gate_hidden)\n        self.gradient_checkpointing = config.gradient_checkpointing\n        self.requires_grad_(True)\n        self.coatnet.set_grad_checkpointing(config.gradient_checkpointing)\n\n    def _fuse(self, mlp: nn.Module, first: Tensor, second: Tensor) -> Tensor:\n        if self.gradient_checkpointing and self.training and torch.is_grad_enabled():\n            return checkpoint(mlp, first, second, use_reentrant=False)\n        return mlp(first, second)\n\n    def forward_features(self, image: Tensor) -> Tensor:\n        if image.ndim != 4 or tuple(image.shape[1:]) != (3, 384, 384):\n            raise ValueError("Both backbones require [N, 3, 384, 384] preprocessed triplets")\n        # All three acquired-slice channels reach both trunks unchanged.\n        # Feed the identical image object; no middle-slice selection/replication.\n        x, y = self.resnet50(image)\n        z = self.coatnet.stages[0](self.coatnet.stem(image))\n        z = self.coatnet.stages[1](z)  # S2 convolution, [N,256,48,48]\n        z = z + self._fuse(self.f1, x, z)  # F1 input order is (X,Z)\n        v = self.coatnet.stages[2](z)  # S3 receives the UPDATED Z\n        v = self.coatnet.stages[3](v)  # S4 transformer, [N,1024,12,12]\n        v = v + self._fuse(self.f2, v, y)  # F2 input order is (V,Y)\n        return self.coatnet.norm(v)\n\n    def forward(self, image: Tensor) -> Tensor:\n        return self.coatnet.forward_head(self.forward_features(image))\n\n\nclass RaptorClassifier(nn.Module):\n    """Original Raptor attention-MIL head, retaining checkpoint parameter names."""\n\n    def __init__(self, backbone: nn.Module, F_dim: int, n: int = 12, drop: float = 0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x: Tensor) -> Tensor:\n        if x.ndim != 5 or x.shape[0] == 0 or x.shape[1] == 0:\n            raise ValueError("Raptor expects a nonempty [B,K,3,384,384] triplet bag")\n        b, k = x.shape[:2]\n        return self.backbone(x.flatten(0, 1)).reshape(b, k, -1)\n\n    def head(self, feats: Tensor) -> Tensor:\n        h = self.norm(feats)\n        a = torch.softmax(self.att(h), dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        return (pooled * self.clsW).sum(-1) + self.clsb\n\n    def forward(self, x: Tensor) -> Tensor:\n        return self.head(self.encode(x))\n\n\ndef _checkpoint(source: str | Path | Mapping[str, Any]) -> Mapping[str, Any]:\n    return source if isinstance(source, Mapping) else torch.load(\n        source, map_location="cpu", weights_only=True,\n    )\n\n\ndef build_raptor_v5_resnet50(\n    raptor_checkpoint: str | Path | Mapping[str, Any],\n    resnet50_checkpoint: str | Path | Mapping[str, Any] | None = None,\n    config: FusionConfig = FusionConfig(),\n    *, resnet50_weights: str | None = "IMAGENET1K_V2",\n) -> RaptorClassifier:\n    """Strict-load both pretrained parents, add zero residuals, unfreeze all.\n\n    Raptor checkpoint: {\'model\': state_dict, \'arch\': ..., \'res\': 384}.\n    ResNet50 defaults to torchvision ImageNet-1K V2 pretrained weights.\n    For offline use, supply the full torchvision state dict (or {\'state_dict\': ...}).\n    Explicit resnet50_weights=None is for random-weight architecture tests.\n    Load and resume a *fused* checkpoint using load_fused_checkpoint instead.\n    """\n    import timm\n\n    ck = _checkpoint(raptor_checkpoint)\n    if ck.get("arch", COATNET_ARCH) != COATNET_ARCH or int(ck.get("res", 384)) != 384:\n        raise ValueError("Parent checkpoint must be Raptor v5 CoAtNet RMLP-2 at res=384")\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    model = RaptorClassifier(coatnet, F_dim=coatnet.num_features, drop=config.head_dropout)\n    model.load_state_dict(ck["model"], strict=True)\n    resnet = ResNet50Encoder(resnet50_checkpoint, weights=resnet50_weights)\n    model.backbone = FusedCoAtNet(coatnet, resnet, config)\n    model.fusion_config = config\n    model.requires_grad_(True)\n    return model\n\n\ndef optimizer_groups(model: RaptorClassifier, backbone_lr: float = 1e-5,\n                     fusion_lr: float = 1e-4, head_lr: float = 1e-4) -> list[dict[str, Any]]:\n    """All parameters, exactly once, with both backbones unfrozen."""\n    if not isinstance(model.backbone, FusedCoAtNet):\n        raise TypeError("Expected the fused Raptor model")\n    if any(not p.requires_grad for p in model.parameters()):\n        raise ValueError("Both backbones and the fusion/head must remain unfrozen")\n    groups = {"coatnet": [], "resnet50": [], "fusion": [], "head": []}\n    for name, parameter in model.named_parameters():\n        if name.startswith("backbone.coatnet."):\n            group = "coatnet"\n        elif name.startswith("backbone.resnet50."):\n            group = "resnet50"\n        elif name.startswith(("backbone.f1.", "backbone.f2.")):\n            group = "fusion"\n        else:\n            group = "head"\n        groups[group].append(parameter)\n    rates = dict(coatnet=backbone_lr, resnet50=backbone_lr, fusion=fusion_lr, head=head_lr)\n    return [{"name": name, "params": parameters, "lr": rates[name]}\n            for name, parameters in groups.items()]\n\n\ndef fused_checkpoint(model: RaptorClassifier) -> dict[str, Any]:\n    """Serialize model/config; trainers can add optimizer, scaler and epoch."""\n    return {"format": FUSED_FORMAT, "arch": COATNET_ARCH, "res": 384,\n            "labels": list(LABELS), "fusion_config": asdict(model.fusion_config),\n            "model": model.state_dict()}\n\n\ndef load_fused_checkpoint(source: str | Path | Mapping[str, Any]) -> RaptorClassifier:\n    import timm\n\n    ck = _checkpoint(source)\n    if (ck.get("format") != FUSED_FORMAT or ck.get("arch") != COATNET_ARCH\n            or ck.get("res") != 384 or ck.get("labels") != list(LABELS)):\n        raise ValueError("Unsupported fused checkpoint architecture/input/label contract")\n    config = FusionConfig(**ck["fusion_config"])\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    backbone = FusedCoAtNet(coatnet, ResNet50Encoder(weights=None), config)\n    model = RaptorClassifier(backbone, F_dim=backbone.num_features, drop=config.head_dropout)\n    model.fusion_config = config\n    model.load_state_dict(ck["model"], strict=True)\n    model.requires_grad_(True)\n    return model\n', 'preprocess_six_slot_raptor.py': '"""Six MRI slots with the saved Raptor V5 image preprocessing contract.\n\nRuns on Kaggle. Outputs fully normalized float16 [K,3,384,384] study bags.\nNo DICOM data is downloaded to the local development computer.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport shutil\nimport threading\nimport time\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport torch\nimport torch.nn.functional as F\nfrom pydicom.pixels import apply_modality_lut\n\n# Six-slot extension; original V5 used five slots with 18/14/12/8/12 references.\nSLOTS = [\n    ("SAG_FLUID_FS", "Sagittal", 1, 1, 18),\n    ("COR_FLUID_FS", "Coronal", 1, 1, 12),\n    ("AX_FLUID_FS", "Axial", 1, 1, 12),\n    ("SAG_FLUID_NOFS", "Sagittal", 1, 0, 14),\n    ("COR_T1", "Coronal", 0, 0, 8),\n    ("SAG_T1", "Sagittal", 0, 0, 8),\n]\nMEAN = torch.tensor([.485, .456, .406]).view(1, 3, 1, 1)\nSTD = torch.tensor([.229, .224, .225]).view(1, 3, 1, 1)\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nRECIPE = {\n    "format": "rsna_six_slot_raptor_v5_v1", "slots": SLOTS,\n    "source_span": [.02, .98], "intensity_percentiles": [2, 98],\n    "intensity_reference": "original evenly spaced per-series reference slices",\n    "crop_mm": 140, "cache_resolution": 336, "model_resolution": 384,\n    "resize": "cv2.INTER_AREA to 336; torch bilinear align_corners=False to 384",\n    "quantization": "uint8 after 336 resize, before /255 and 384 interpolation",\n    "mean": [.485, .456, .406], "std": [.229, .224, .225],\n    "dtype": "float16", "normalized": True, "source_budget": 64,\n    "triplets": "sliding adjacent sampled slices in concatenated slot order, like parent V5",\n    "missing_slots": "absent, never substitute a different category or fabricate pixels",\n    "selector": "dicom_weight_type; positive fat-sat header evidence overrides metadata",\n    "labels": "annotations are separate from pixels; gold74 excluded from training split",\n}\nRECIPE_SHA = hashlib.sha256(json.dumps(RECIPE, sort_keys=True).encode()).hexdigest()\nLOCK = threading.Lock()\n\n\ndef json_write(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n    tmp.replace(path)\n\n\ndef allocate(capacities, weights, target=64):\n    capacities = np.asarray(capacities, dtype=np.int64)\n    weights = np.asarray(weights, dtype=np.float64)\n    ideal = target * weights / weights.sum()\n    quotas = np.floor(ideal).astype(np.int64)\n    for i in np.argsort(-(ideal-quotas), kind="stable")[:target-int(quotas.sum())]:\n        quotas[i] += 1\n    quotas = np.minimum(quotas, capacities)\n    deficit = np.zeros(len(quotas))\n    wanted = min(target, int(capacities.sum()))\n    while quotas.sum() < wanted:\n        available = quotas < capacities\n        active = np.where(available, weights, 0)\n        deficit[available] += active[available]/active.sum()\n        choice = int(np.argmax(np.where(available, deficit, -np.inf)))\n        quotas[choice] += 1\n        deficit[choice] -= 1\n    return quotas\n\n\ndef flag(value):\n    try:\n        value = float(value)\n        return int(value) if value in (0, 1) else None\n    except (ValueError, TypeError):\n        return None\n\n\ndef weight_type(h):\n    desc = re.sub(r"[_\\-.]", " ", f"{getattr(h, \'SeriesDescription\', \'\')} {getattr(h, \'SequenceName\', \'\')}".lower())\n    t1 = bool(re.search(r"\\bt1\\b|\\bt1w\\b", desc))\n    t2 = bool(re.search(r"\\bt2\\b|\\bt2w\\b", desc))\n    pdw = bool(re.search(r"\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens", desc))\n    tr, te = float(getattr(h, \'RepetitionTime\', \'nan\')), float(getattr(h, \'EchoTime\', \'nan\'))\n    if t1 and not t2 and not pdw:\n        return "T1"\n    if t2 and not pdw:\n        return "T2"\n    if pdw:\n        return "PD"\n    if "GR" in str(getattr(h, "ScanningSequence", "")).upper():\n        return "GRE"\n    return "T1" if tr < 800 else "T2" if te > 60 else "PD" if tr >= 800 else "UNK"\n\n\ndef classify(row, directory):\n    files = sorted(directory.glob("*.dcm"))\n    if not files:\n        return None\n    h = pydicom.dcmread(files[len(files)//2], stop_before_pixels=True)\n    kind = weight_type(h)\n    # In this competition CSV, Fluid_Sensitive and Fat_Suppression are\n    # effectively coupled. PD/T2 without FS must be recovered from the header,\n    # as in the original recovered six-slot classifier.\n    fluid = int(kind in ("PD", "T2")) if kind != "UNK" else flag(row.get("Fluid_Sensitive"))\n    fs = flag(row.get("Fat_Suppression"))\n    desc = re.sub(r"[_\\-.]", " ", f"{getattr(h, \'SeriesDescription\', \'\')} {getattr(h, \'SequenceName\', \'\')}".lower())\n    options = getattr(h, "ScanOptions", [])\n    options = options.split(\'\\\\\') if isinstance(options, str) else options\n    opts = set(str(v).upper() for v in options)\n    header_fs = bool(re.search(r"\\bfs\\b|fat\\s*sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b|smart fat|\\bwater\\b", desc)) or bool(opts & {"FS", "FATSAT", "FAT_SAT", "FSAT"})\n    if header_fs:\n        fs = 1\n    elif fs is None:\n        fs = 0\n    return {"uid": str(row["SeriesInstanceUID"]), "plane": row["Anatomical_Plane"],\n            "fluid": fluid, "fs": fs, "weight": kind, "files": files,\n            "description": str(getattr(h, "SeriesDescription", ""))}\n\n\ndef order_files(series):\n    records, sop_uids = [], set()\n    for file in series["files"]:\n        h = pydicom.dcmread(file, stop_before_pixels=True)\n        sop = str(h.SOPInstanceUID)\n        if sop in sop_uids:\n            raise ValueError(f"Duplicate SOPInstanceUID: {file}")\n        sop_uids.add(sop)\n        if str(h.SeriesInstanceUID) != series["uid"]:\n            raise ValueError(f"Series UID mismatch: {file}")\n        iop, ipp = getattr(h, "ImageOrientationPatient", None), getattr(h, "ImagePositionPatient", None)\n        if iop is not None and ipp is not None and len(iop) == 6:\n            position = float(np.dot(np.asarray(ipp, float), np.cross(np.asarray(iop[:3], float), np.asarray(iop[3:], float))))\n        else:\n            position = float(h.InstanceNumber)\n        if not np.isfinite(position):\n            raise ValueError(f"Nonfinite slice coordinate: {file}")\n        spacing = float(getattr(h, "PixelSpacing", [.5])[0])\n        records.append((position, str(file), spacing))\n    records.sort(key=lambda r: (r[0], r[1]))\n    good_spacing = [r[2] for r in records if np.isfinite(r[2]) and r[2] > 0]\n    median = float(np.median(good_spacing)) if good_spacing else .5\n    return [(file, spacing if np.isfinite(spacing) and spacing > 0 else median) for _, file, spacing in records]\n\n\ndef pixels(file):\n    d = pydicom.dcmread(file)\n    a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n    if a.ndim != 2 or not np.isfinite(a).all():\n        raise ValueError(f"Invalid acquired pixels: {file}")\n    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":\n        a = a.max()-a\n    return a\n\n\ndef crop_plane(a, spacing, low, high):\n    a = np.clip((a-low)/(high-low+1e-6), 0, 1)\n    h, w = a.shape\n    size = min(int(round(140/max(spacing, .001))), min(h, w))\n    y, x = (h-size)//2, (w-size)//2\n    a = cv2.resize(a[y:y+size, x:x+size], (336, 336), interpolation=cv2.INTER_AREA)\n    return (a*255).astype(np.uint8)\n\n\ndef model_ready(planes):\n    # Preserve parent\'s /255, interpolation, then channel-specific normalization.\n    volume = np.stack(planes)\n    source = torch.from_numpy(volume.astype(np.float32)/255)\n    resized = F.interpolate(source[:, None], size=(384, 384), mode="bilinear", align_corners=False)[:, 0]\n    x = torch.stack([resized[:-2], resized[1:-1], resized[2:]], dim=1)\n    return ((x-MEAN)/STD).numpy().astype(np.float16)\n\n\ndef prepare(uid, rows, series_root):\n    candidates, warnings = [], []\n    for row in rows:\n        s = classify(row, series_root/uid/str(row["SeriesInstanceUID"]))\n        if s is not None:\n            candidates.append(s)\n    used, pools, selection = set(), [], []\n    for name, plane, fluid, fs, reference_count in SLOTS:\n        matching = [s for s in candidates if s["uid"] not in used and s["plane"] == plane\n                    and s["fluid"] == fluid and s["fs"] == fs\n                    and (fluid == 1 or s["weight"] == "T1")]\n        # Retain metadata row priority, matching the Raptor selection convention.\n        if not matching:\n            pools.append(None)\n            selection.append({"slot": name, "present": False})\n            continue\n        series = matching[0]\n        used.add(series["uid"])\n        files = order_files(series)\n        lo = int(len(files)*.02)\n        hi = max(int(len(files)*.98)-1, lo)\n        refs = np.linspace(lo, hi, reference_count).round().astype(int)\n        pools.append((files, lo, hi, refs))\n        selection.append({"slot": name, "present": True, "series_uid": series["uid"],\n                          "description": series["description"], "weight": series["weight"]})\n    capacities = [0 if p is None else p[2]-p[1]+1 for p in pools]\n    quotas = allocate(capacities, [s[4] for s in SLOTS])\n    planes, slot_ids, sources, intensity = [], [], [], []\n    for slot, (pool, count) in enumerate(zip(pools, quotas)):\n        if pool is None or not count:\n            continue\n        files, lo, hi, refs = pool\n        picks = lo + np.linspace(0, hi-lo, int(count)).round().astype(int)\n        arrays = {int(i): pixels(files[int(i)][0]) for i in sorted(set(refs)|set(picks))}\n        low, high = np.percentile(np.concatenate([arrays[int(i)].ravel() for i in refs]), [2, 98])\n        intensity.append({"slot": slot, "p2": float(low), "p98": float(high), "reference_indices": refs.tolist()})\n        for i in picks:\n            file, spacing = files[int(i)]\n            planes.append(crop_plane(arrays[int(i)], spacing, low, high))\n            slot_ids.append(slot)\n            sources.append({"slot": slot, "file": Path(file).name, "series_uid": selection[slot]["series_uid"], "native_index": int(i)})\n    if len(planes) < 3:\n        raise ValueError("Fewer than three acquired slices across the six categories")\n    x = model_ready(planes)\n    ids = np.asarray(slot_ids, dtype=np.int8)\n    triplet_slots = np.stack([ids[:-2], ids[1:-1], ids[2:]], 1)\n    pure = (triplet_slots[:, 0] == triplet_slots[:, 1]) & (triplet_slots[:, 1] == triplet_slots[:, 2])\n    audit = {"study_uid": uid, "recipe_sha256": RECIPE_SHA, "selection": selection,\n             "capacities": capacities, "quotas": quotas.tolist(), "source_slices": len(planes),\n             "windows": len(x), "cross_series_windows": int((~pure).sum()),\n             "source_files": sources, "intensity": intensity, "warnings": warnings}\n    if not np.isfinite(x).all() or tuple(x.shape[1:]) != (3, 384, 384):\n        raise ValueError("Nonfinite or invalid model input")\n    return {"images": x, "triplet_slot_ids": triplet_slots, "same_series": pure,\n            "slot_present": np.asarray([s["present"] for s in selection], bool),\n            "audit_json": np.asarray(json.dumps(audit, allow_nan=False))}, audit\n\n\ndef save_npz(path, data):\n    tmp = path.with_suffix(".npz.tmp")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:\n        for name, array in data.items():\n            with z.open(name+".npy", "w", force_zip64=True) as member:\n                np.lib.format.write_array(member, np.asarray(array), allow_pickle=False)\n    tmp.replace(path)\n\n\ndef find_root():\n    for p in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection"):\n        if (Path(p)/"train_series.csv").is_file():\n            return Path(p)\n    raise FileNotFoundError("Attach rsna-knee-abnormality-detection competition data")\n\n\ndef run(shard=0, shards=32, limit=None, root=None, output=None, workers=2):\n    print(json.dumps({\'phase\':\'input_setup\',\'shard\':shard}), flush=True)\n    torch.set_num_threads(1)\n    cv2.setNumThreads(1)\n    root = find_root() if root is None else Path(root)\n    output = Path(output or f"/kaggle/working/six_slot_raptor_shard_{shard:02d}")\n    output.mkdir(parents=True, exist_ok=True)\n    series = pd.read_csv(root/"train_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})\n    labels = pd.read_csv(root/"train.csv", dtype={"StudyInstanceUID": str})\n    ids = sorted(labels.StudyInstanceUID.unique())[shard::shards]\n    if limit:\n        ids = ids[:limit]\n    groups = {str(uid): g.to_dict("records") for uid, g in series.groupby("StudyInstanceUID", sort=False)}\n    series_root = next((root/p for p in ("train_series", "train_images", "train") if (root/p).is_dir()), None)\n    if series_root is None:\n        raise FileNotFoundError("No competition training DICOM directory")\n    json_write(output/"recipe.json", dict(RECIPE, sha256=RECIPE_SHA))\n    labels[labels.StudyInstanceUID.isin(ids)].to_csv(output/"labels_official.csv", index=False)\n    gold = set(labels.loc[labels[LABELS].notna().all(axis=1), "StudyInstanceUID"])\n    # Never recursively scan the competition\'s millions of DICOM files to find\n    # a small annotation CSV; use the two supported attached-dataset layouts.\n    jev = next((p/\'gold74_eval_set.csv\' for p in (\n        Path(\'/kaggle/input/datasets/riadmohamed42/jev-knee-labels\'),\n        Path(\'/kaggle/input/jev-knee-labels\')) if (p/\'gold74_eval_set.csv\').is_file()), None)\n    if jev is None:\n        raise FileNotFoundError(\'Attach riadmohamed42/jev-knee-labels: gold74 metadata is required\')\n    if jev is not None:\n        gold |= set(pd.read_csv(jev, dtype={"StudyInstanceUID": str}).StudyInstanceUID)\n        final2 = jev.parent/"FINAL2_labels_scores.csv"\n        if final2.is_file():\n            table = pd.read_csv(final2, dtype={"StudyInstanceUID": str})\n            table[table.StudyInstanceUID.isin(ids)].to_csv(output/"labels_jev_final2.csv", index=False)\n    pd.DataFrame({"StudyInstanceUID": ids, "split": ["held_out_gold" if uid in gold else "train" for uid in ids]}).to_csv(output/"splits.csv", index=False)\n    print(json.dumps({\'phase\':\'metadata_ready\',\'shard\':shard,\'studies\':len(ids),\'series_root\':str(series_root)}), flush=True)\n    start = time.time()\n    results, failures = [], []\n    def heartbeat(state="running"):\n        value = {"state": state, "shard": shard, "shards": shards, "expected_studies": len(ids),\n                 "completed_studies": len(results), "failed_studies": len(failures),\n                 "elapsed_seconds": round(time.time()-start, 1), "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n                 "output_bytes": sum(r["bytes"] for r in results), "recipe_sha256": RECIPE_SHA}\n        json_write(output/"progress.json", value)\n        print(json.dumps(value), flush=True)\n    def one(uid):\n        file = output/(uid+".npz")\n        # Reserve worst-case remaining uncompressed bag size before processing.\n        if shutil.disk_usage(output).free < 512*1024**2:\n            raise OSError("Less than 512 MiB free: stop before corrupting output")\n        data, audit = prepare(uid, groups[uid], series_root)\n        save_npz(file, data)\n        # Re-open and check the exact saved model tensor.\n        with np.load(file, allow_pickle=False) as saved:\n            if not np.array_equal(saved["images"], data["images"]):\n                raise ValueError("Saved tensor round-trip mismatch")\n        digest = hashlib.sha256()\n        with file.open("rb") as f:\n            for block in iter(lambda: f.read(8*1024**2), b""):\n                digest.update(block)\n        return {"study_uid": uid, "file": file.name, "windows": audit["windows"],\n                "cross_series_windows": audit["cross_series_windows"], "slot_present": data["slot_present"].tolist(),\n                "bytes": file.stat().st_size, "sha256": digest.hexdigest()}\n    heartbeat()\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        futures = {pool.submit(one, uid): uid for uid in ids}\n        for future in as_completed(futures):\n            uid = futures[future]\n            try:\n                row = future.result()\n                results.append(row)\n                with (output/"index.jsonl").open("a") as f:\n                    f.write(json.dumps(row)+"\\n")\n            except Exception as exc:\n                failure = {"study_uid": uid, "error": f"{type(exc).__name__}: {exc}"}\n                failures.append(failure)\n                with (output/"failures.jsonl").open("a") as f:\n                    f.write(json.dumps(failure)+"\\n")\n                print("STUDY_FAILED "+json.dumps(failure), flush=True)\n            heartbeat()\n    manifest = {"recipe_sha256": RECIPE_SHA, "shard": shard, "shards": shards,\n                "expected_uids": ids, "completed": results, "failures": failures,\n                "complete": len(results) == len(ids) and not failures}\n    json_write(output/"manifest.json", manifest)\n    heartbeat("complete" if manifest["complete"] else "incomplete")\n    if failures:\n        raise RuntimeError(f"Shard incomplete: {len(failures)} failed studies; see failures.jsonl")\n    return manifest\n\n\nif __name__ == "__main__":\n    p = argparse.ArgumentParser()\n    p.add_argument("--shard", type=int, default=0)\n    p.add_argument("--shards", type=int, default=32)\n    p.add_argument("--limit", type=int)\n    p.add_argument("--root")\n    p.add_argument("--output")\n    a = p.parse_args()\n    run(**vars(a))\n', 'preprocess_series_safe_384.py': '"""Six-slot direct-384 exports with provably single-series triplets."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom preprocess_six_slot_raptor import classify, pixels, save_npz, find_root, json_write\nimport pydicom\n\nSLOTS = [\n    (\'SAG_FLUID_FS\',\'Sagittal\',1,1,18),\n    (\'SAG_FLUID_NOFS\',\'Sagittal\',1,0,14),\n    (\'COR_FLUID_FS\',\'Coronal\',1,1,12),\n    (\'COR_T1\',\'Coronal\',0,0,8),\n    (\'AX_FLUID_FS\',\'Axial\',1,1,12),\n    (\'SAG_T1\',\'Sagittal\',0,0,8),\n]\nLABELS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\n          \'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nMEAN = np.asarray([.485,.456,.406],np.float32).reshape(1,3,1,1)\nSTD = np.asarray([.229,.224,.225],np.float32).reshape(1,3,1,1)\nRECIPE = dict(format=\'rsna_series_safe_384_v1\',slots=SLOTS,source_span=[.02,.98],\n    depth_indices=\'lo=floor(N*.02); hi=max(floor(N*.98)-1,lo), inclusive\',\n    intensity_percentiles=[2,98],intensity_reference=\'the actual unique sampled slices in each selected series\',\n    sampling=\'evenly spaced unique physical-order slices, up to slot allocation; no redistribution or duplication\',\n    source_budget=72,crop_mm=140,crop=\'center rectangle using row/column PixelSpacing separately, clamp to acquired field\',\n    resolution=384,model_resolution=384,cache_resolution=384,resize=\'direct cv2.INTER_AREA to 384; no intermediate resize\',\n    quantization=\'floor(clipped_scaled_resized*255) uint8\',slices_dtype=\'uint8\',dtype=\'float16\',normalized=True,\n    mean=[.485,.456,.406],std=[.229,.224,.225],triplets=\'sliding three neighboring sampled slices WITHIN one SeriesInstanceUID\',\n    missing_slots=\'absent; never substitute, pad or duplicate slices\',selector=\'dicom_weight_type; positive fat-sat header evidence overrides metadata\',\n    physical_order=\'common IOP slice normal dot IPP; fail on missing/inconsistent geometry\',\n    labels=\'Gold74 excluded from training split; held_out_gold separate; HYBRID scores separate from pixels\')\nRECIPE_SHA=hashlib.sha256(json.dumps(RECIPE,sort_keys=True).encode()).hexdigest()\n\n\ndef physical_order(series,study_uid):\n    records=[]; sop_uids=set(); normal=None\n    for file in series[\'files\']:\n        h=pydicom.dcmread(file,stop_before_pixels=True)\n        if str(h.SeriesInstanceUID)!=series[\'uid\'] or str(h.StudyInstanceUID)!=study_uid:\n            raise ValueError(\'DICOM series/study ownership mismatch\')\n        sop=str(h.SOPInstanceUID)\n        if sop in sop_uids:raise ValueError(\'Duplicate acquired SOPInstanceUID\')\n        sop_uids.add(sop)\n        iop=np.asarray(getattr(h,\'ImageOrientationPatient\',[]),float)\n        ipp=np.asarray(getattr(h,\'ImagePositionPatient\',[]),float)\n        spacing=np.asarray(getattr(h,\'PixelSpacing\',[]),float)\n        if iop.shape!=(6,) or ipp.shape!=(3,) or spacing.shape!=(2,) or not np.isfinite(np.r_[iop,ipp,spacing]).all() or (spacing<=0).any():\n            raise ValueError(\'Missing/invalid physical geometry or PixelSpacing\')\n        candidate=np.cross(iop[:3],iop[3:])\n        if np.linalg.norm(candidate)<.99:raise ValueError(\'Invalid slice normal\')\n        candidate=candidate/np.linalg.norm(candidate)\n        if normal is None:normal=candidate\n        if np.dot(normal,candidate)<.999:raise ValueError(\'Inconsistent slice orientations\')\n        records.append(dict(file=str(file),spacing=spacing.tolist(),position=float(np.dot(ipp,normal)),sop_uid=sop))\n    records.sort(key=lambda r:(r[\'position\'],r[\'sop_uid\']))\n    if len({round(r[\'position\'],5) for r in records})!=len(records):\n        raise ValueError(\'Duplicate physical slice positions\')\n    return records\n\n\ndef sample_indices(count,allocation):\n    if count<1:raise ValueError(\'Empty MRI series\')\n    lo=int(count*.02); hi=max(int(count*.98)-1,lo)\n    chosen=np.linspace(lo,hi,min(allocation,hi-lo+1)).round().astype(np.int32)\n    if len(set(chosen.tolist()))!=len(chosen):raise ValueError(\'Duplicate sampled index\')\n    return chosen\n\n\ndef crop_plane(array,spacing,low,high):\n    scaled=np.clip((array-low)/(high-low+1e-6),0,1)\n    height,width=scaled.shape\n    ch=min(max(1,int(round(140/spacing[0]))),height)\n    cw=min(max(1,int(round(140/spacing[1]))),width)\n    top,left=(height-ch)//2,(width-cw)//2\n    resized=cv2.resize(scaled[top:top+ch,left:left+cw],(384,384),interpolation=cv2.INTER_AREA)\n    return (resized*255).astype(np.uint8)\n\n\ndef normalized_triplets(slices,indices):\n    x=slices[indices].astype(np.float32)/np.float32(255.)\n    return ((x-MEAN)/STD).astype(np.float16)\n\n\ndef validate_arrays(data):\n    slices,indices,x=data[\'slices_uint8\'],data[\'triplet_indices\'],data[\'images\']\n    if slices.dtype!=np.uint8 or slices.ndim!=3 or slices.shape[1:]!=(384,384):raise ValueError(\'Invalid stored slices\')\n    if not len(indices) or indices.shape!=(len(x),3) or (indices<0).any() or (indices>=len(slices)).any():raise ValueError(\'Invalid triplet indices\')\n    owners=data[\'slice_series_uids\'][indices]\n    if not np.all(owners==owners[:,:1]):raise ValueError(\'CROSS-SERIES TRIPLET\')\n    if not np.all(data[\'same_series\']) or not np.all(data[\'triplet_slot_ids\']==data[\'slice_slot_ids\'][indices]):raise ValueError(\'Triplet provenance mismatch\')\n    if not np.all(owners[:,0]==data[\'triplet_series_uids\']):raise ValueError(\'Triplet series UID mismatch\')\n    if x.dtype!=np.float16 or x.shape[1:]!=(3,384,384) or not np.isfinite(x).all():raise ValueError(\'Invalid normalized images\')\n    if not np.array_equal(normalized_triplets(slices,indices),x):raise ValueError(\'Normalized images differ from stored slices\')\n\n\ndef prepare(uid,rows,series_root):\n    candidates=[]\n    for row in rows:\n        candidate=classify(row,series_root/uid/str(row[\'SeriesInstanceUID\']))\n        if candidate is not None:candidates.append(candidate)\n    used=set(); selection=[]; slices=[]; sources=[]; slot_ids=[]; series_uids=[]; triplets=[]; intensity=[]\n    for slot,(name,plane,fluid,fs,allocation) in enumerate(SLOTS):\n        matches=[s for s in candidates if s[\'uid\'] not in used and s[\'plane\']==plane and\n                 s[\'fluid\']==fluid and s[\'fs\']==fs and (fluid or s[\'weight\']==\'T1\')]\n        if not matches:\n            selection.append(dict(slot=name,present=False));continue\n        series=matches[0];used.add(series[\'uid\'])\n        files=physical_order(series,uid); picks=sample_indices(len(files),allocation)\n        arrays=[pixels(files[int(i)][\'file\']) for i in picks]\n        low,high=np.percentile(np.concatenate([a.ravel() for a in arrays]),[2,98])\n        offset=len(slices)\n        for native_index,array in zip(picks,arrays):\n            record=files[int(native_index)]\n            slices.append(crop_plane(array,record[\'spacing\'],low,high))\n            slot_ids.append(slot);series_uids.append(series[\'uid\'])\n            sources.append(dict(slot=slot,series_uid=series[\'uid\'],file=Path(record[\'file\']).name,\n                                sop_uid=record[\'sop_uid\'],native_index=int(native_index),position=record[\'position\'],spacing=record[\'spacing\']))\n        triplets.extend([[offset+i,offset+i+1,offset+i+2] for i in range(max(0,len(picks)-2))])\n        selection.append(dict(slot=name,present=True,series_uid=series[\'uid\'],description=series[\'description\'],\n                              weight=series[\'weight\'],allocation=allocation,sampled_count=len(picks)))\n        intensity.append(dict(slot=slot,p2=float(low),p98=float(high),reference_indices=picks.tolist()))\n    if not triplets:raise ValueError(\'No series has three eligible sampled slices\')\n    slices=np.stack(slices); indices=np.asarray(triplets,np.int32); slot_ids=np.asarray(slot_ids,np.int8)\n    series_uids=np.asarray(series_uids)\n    x=normalized_triplets(slices,indices)\n    audit=dict(study_uid=uid,recipe_sha256=RECIPE_SHA,selection=selection,source_slices=len(slices),windows=len(x),\n               cross_series_windows=0,source_files=sources,intensity=intensity)\n    data=dict(slices_uint8=slices,slice_slot_ids=slot_ids,slice_series_uids=series_uids,\n              triplet_indices=indices,triplet_series_uids=series_uids[indices[:,0]],\n              images=x,triplet_slot_ids=slot_ids[indices],same_series=np.ones(len(x),bool),\n              slot_present=np.asarray([s[\'present\'] for s in selection],bool),audit_json=np.asarray(json.dumps(audit,allow_nan=False)))\n    validate_arrays(data)\n    return data,audit\n\n\ndef run(shard=0,shards=32,limit=None,root=None,output=None,workers=4):\n    torch.set_num_threads(1);cv2.setNumThreads(1)\n    root=Path(root) if root else find_root()\n    output=Path(output or f\'/kaggle/working/series_safe_384_shard_{shard:02d}\')\n    output.mkdir(parents=True,exist_ok=True)\n    series=pd.read_csv(root/\'train_series.csv\',dtype={\'StudyInstanceUID\':str,\'SeriesInstanceUID\':str})\n    labels=pd.read_csv(root/\'train.csv\',dtype={\'StudyInstanceUID\':str})\n    jev=next((p for p in [Path(\'/kaggle/input/datasets/riadmohamed42/jev-knee-labels\'),Path(\'/kaggle/input/jev-knee-labels\')]\n              if (p/\'gold74_eval_set.csv\').is_file()),None)\n    if jev is None:raise FileNotFoundError(\'Gold74 exclusion requires riadmohamed42/jev-knee-labels\')\n    gold_labels=pd.read_csv(jev/\'gold74_eval_set.csv\',dtype={\'StudyInstanceUID\':str})\n    gold=set(gold_labels.StudyInstanceUID)\n    official_gold=set(labels.loc[labels[LABELS].notna().all(axis=1),\'StudyInstanceUID\'])\n    if len(gold)!=74 or not official_gold<=gold:raise ValueError(\'Gold74 metadata mismatch\')\n    all_ids=sorted(labels.StudyInstanceUID.unique());ids=all_ids[shard::shards]\n    if limit is not None:ids=ids[:limit]\n    if not set(gold)<=set(all_ids):raise ValueError(\'Gold74 IDs outside competition corpus\')\n    groups={str(uid):group.to_dict(\'records\') for uid,group in series.groupby(\'StudyInstanceUID\',sort=False)}\n    series_root=next((root/p for p in (\'train_series\',\'train_images\',\'train\') if (root/p).is_dir()),None)\n    if series_root is None:raise FileNotFoundError(\'Training DICOM directory not found\')\n    json_write(output/\'recipe.json\',{**RECIPE,\'sha256\':RECIPE_SHA})\n    split=pd.DataFrame(dict(StudyInstanceUID=ids,split=[\'held_out_gold\' if uid in gold else \'train\' for uid in ids]))\n    split.to_csv(output/\'splits.csv\',index=False)\n    labels[labels.StudyInstanceUID.isin(ids)].to_csv(output/\'labels_official.csv\',index=False)\n    gold_labels[gold_labels.StudyInstanceUID.isin(ids)].to_csv(output/\'labels_gold74.csv\',index=False)\n    train_ids=set(ids)-gold\n    hybrid=pd.read_csv(jev/\'HYBRID_labels_scores.csv\',dtype={\'StudyInstanceUID\':str})\n    hybrid=hybrid[hybrid.StudyInstanceUID.isin(train_ids)]\n    if set(hybrid.StudyInstanceUID)!=train_ids or hybrid.StudyInstanceUID.duplicated().any() or not np.isfinite(hybrid[LABELS].to_numpy()).all():\n        raise ValueError(\'Missing or invalid non-gold HYBRID targets\')\n    hybrid.to_csv(output/\'labels_hybrid_train.csv\',index=False)\n    results=[];failures=[];start=time.time()\n    def progress(state=\'running\'):\n        record=dict(state=state,shard=shard,shards=shards,expected_studies=len(ids),completed_studies=len(results),\n                    failed_studies=len(failures),output_bytes=sum(r[\'bytes\'] for r in results),\n                    elapsed_seconds=round(time.time()-start,1),recipe_sha256=RECIPE_SHA,\n                    updated_utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()))\n        json_write(output/\'progress.json\',record);print(json.dumps(record),flush=True)\n    def one(uid):\n        if shutil.disk_usage(output).free<512*1024**2:raise OSError(\'Insufficient disk space\')\n        data,audit=prepare(uid,groups[uid],series_root)\n        split_name=\'held_out_gold\' if uid in gold else \'train\'\n        folder=output/split_name;folder.mkdir(exist_ok=True)\n        file=folder/(uid+\'.npz\');save_npz(file,data)\n        with np.load(file,allow_pickle=False) as saved:\n            validate_arrays(saved)\n            if not np.array_equal(saved[\'slices_uint8\'],data[\'slices_uint8\']):raise ValueError(\'Saved uint8 slices mismatch\')\n        with file.open(\'rb\') as stream:digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\n        return dict(study_uid=uid,file=str(file.relative_to(output)).replace(\'\\\\\',\'/\'),split=split_name,\n                    windows=audit[\'windows\'],source_slices=audit[\'source_slices\'],cross_series_windows=0,\n                    slot_present=data[\'slot_present\'].tolist(),bytes=file.stat().st_size,sha256=digest)\n    progress()\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        futures={pool.submit(one,uid):uid for uid in ids}\n        for future in as_completed(futures):\n            try:\n                row=future.result();results.append(row)\n                with (output/\'index.jsonl\').open(\'a\') as stream:stream.write(json.dumps(row)+\'\\n\')\n            except Exception as exc:\n                failure=dict(study_uid=futures[future],error=f\'{type(exc).__name__}: {exc}\');failures.append(failure)\n                with (output/\'failures.jsonl\').open(\'a\') as stream:stream.write(json.dumps(failure)+\'\\n\')\n                print(\'STUDY_FAILED \'+json.dumps(failure),flush=True)\n            progress()\n    manifest=dict(recipe_sha256=RECIPE_SHA,shard=shard,shards=shards,expected_uids=ids,completed=results,\n                  failures=failures,complete=len(results)==len(ids) and not failures)\n    json_write(output/\'manifest.json\',manifest);progress(\'complete\' if manifest[\'complete\'] else \'incomplete\')\n    if not manifest[\'complete\']:raise RuntimeError(\'Incomplete shard; inspect failures.jsonl\')\n    return manifest\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--shard\',type=int,default=0);parser.add_argument(\'--shards\',type=int,default=32)\n    parser.add_argument(\'--limit\',type=int);parser.add_argument(\'--output\');args=parser.parse_args();run(**vars(args))\n', 'preprocess_min_overlap_384.py': '"""Same pixel contract as series-safe 384; globally minimize triplet reuse."""\nfrom __future__ import annotations\nimport hashlib\nimport json\nfrom functools import lru_cache\nfrom pathlib import Path\nimport numpy as np\nimport preprocess_series_safe_384 as parent\n\nSLOTS=parent.SLOTS\nPARENT_SHA=parent.RECIPE_SHA\nBASE_VALIDATE=parent.validate_arrays\nTARGET_WEIGHTS=(16,12,10,6,10,6)\nRECIPE={**parent.RECIPE,\n    \'format\':\'rsna_min_overlap_384_v1\',\'parent_recipe_sha256\':PARENT_SHA,\n    \'intensity_reference\':\'UNCHANGED parent reference indices: unique linspace over native 2-98% band, using 18/14/12/8/12/8 per selected series\',\n    \'sampling\':\'up to 180 unique slices; maximize distinct slices used by up to 60 unique same-series windows\',\n    \'source_budget\':180,\'triplet_budget\':60,\'slot_triplet_weights\':TARGET_WEIGHTS,\n    \'allocation\':\'exact dynamic programming: maximize used unique slices, then maximum mutually disjoint subset, then minimize squared slot-weight deviation\',\n    \'triplets\':\'three adjacent slices in each series sampled list; disjoint groups first, overlapping windows only as required; no duplicate windows\',\n    \'window_fallback\':\'keep fewer than 60 if selected series have fewer than 60 unique neighboring three-slice windows; no duplicate padding\',\n    \'selected_series\':\'UNCHANGED one first matching acquired series per six-slot category; no additional series\',\n    \'missing_slots\':\'absent; triplet budget can shift to available selected slots to minimize reuse; no category substitution\',\n    \'sampler_priority\':\'minimize repeated source-slice uses at the attainable triplet count, before matching category weights\'}\nRECIPE_SHA=hashlib.sha256(json.dumps(RECIPE,sort_keys=True).encode()).hexdigest()\n\n\ndef allocate_triplets(capacities,target=60):\n    allocation,optimal=_allocate_cached(tuple(int(c) for c in capacities),int(target))\n    return allocation.copy(),dict(optimal)\n\n\n@lru_cache(maxsize=4096)\ndef _allocate_cached(capacities,target):\n    """Exact optimal allocation for <=6 series and a <=60 triplet bag.\n\n    c native eligible slices support c-2 distinct neighboring windows. With k\n    windows, min(c,3*k) unique slices and min(k,c//3) disjoint windows are\n    simultaneously attainable by series_windows below.\n    """\n    capacities=tuple(int(c) for c in capacities)\n    if len(capacities)!=6 or any(c<0 for c in capacities):raise ValueError(\'Invalid six-slot capacity\')\n    target=min(int(target),sum(max(c-2,0) for c in capacities))\n    if target<1:raise ValueError(\'No eligible three-slice window\')\n    weights=[w if c>=3 else 0 for w,c in zip(TARGET_WEIGHTS,capacities)]\n    denominator=sum(weights)\n    # State is (unique used, mutually disjoint subset, negative mix penalty).\n    states={0:((0,0,0),())}\n    for c,w in zip(capacities,weights):\n        following={}\n        for total,(score,path) in states.items():\n            for k in range(min(max(c-2,0),target-total)+1):\n                candidate=(score[0]+min(c,3*k),score[1]+min(k,c//3),\n                    score[2]-(k*denominator-target*w)**2)\n                old=following.get(total+k)\n                if old is None or candidate>old[0] or (candidate==old[0] and path+(k,)>old[1]):\n                    following[total+k]=(candidate,path+(k,))\n        states=following\n    score,allocation=states[target]\n    return np.asarray(allocation,np.int32),dict(target_triplets=target,optimal_unique_slices=score[0],\n        optimal_disjoint_subset=score[1],minimum_repeated_slice_uses=3*target-score[0])\n\n\ndef series_windows(lo,hi,k):\n    """Return native sample indices, local windows, and a disjoint-subset mask."""\n    c=int(hi-lo+1);k=int(k)\n    if k<1 or c<3 or k>c-2:raise ValueError(\'Invalid series window count\')\n    count=min(c,3*k)\n    picks=np.linspace(lo,hi,count).round().astype(np.int32)\n    if len(np.unique(picks))!=count:raise ValueError(\'Duplicate native sample\')\n    q=min(k,count//3)\n    baseline=set(range(0,3*q,3));starts=set(baseline)\n    usage=np.zeros(count,np.int32)\n    for start in starts:usage[start:start+3]+=1\n    if count%3 and len(starts)<k:\n        tail=count-3;starts.add(tail);usage[tail:tail+3]+=1\n    while len(starts)<k:\n        candidates=[s for s in range(count-2) if s not in starts]\n        # All sources are already covered: minimize concentrated reuse.\n        start=min(candidates,key=lambda s:(int((usage[s:s+3]+1).max()),\n            int(((usage[s:s+3]+1)**2-usage[s:s+3]**2).sum()),s))\n        starts.add(start);usage[start:start+3]+=1\n    ordered=sorted(starts)\n    windows=np.asarray([[s,s+1,s+2] for s in ordered],np.int32)\n    primary=np.asarray([s in baseline for s in ordered],bool)\n    assert (usage>0).all() and len(windows)==k and primary.sum()==min(k,c//3)\n    return picks,windows,primary\n\n\ndef prepare(uid,rows,series_root):\n    candidates=[]\n    for row in rows:\n        candidate=parent.classify(row,series_root/uid/str(row[\'SeriesInstanceUID\']))\n        if candidate is not None:candidates.append(candidate)\n    used=set();selected=[];capacities=[]\n    for slot,(name,plane,fluid,fs,reference_count) in enumerate(SLOTS):\n        matches=[s for s in candidates if s[\'uid\'] not in used and s[\'plane\']==plane and\n            s[\'fluid\']==fluid and s[\'fs\']==fs and (fluid or s[\'weight\']==\'T1\')]\n        if not matches:selected.append(None);capacities.append(0);continue\n        series=matches[0];used.add(series[\'uid\'])\n        files=parent.physical_order(series,uid)\n        refs=parent.sample_indices(len(files),reference_count)\n        lo,hi=int(refs[0]),int(refs[-1])\n        selected.append((series,files,refs,lo,hi));capacities.append(hi-lo+1)\n    allocation,optimal=allocate_triplets(capacities)\n    selection=[];slices=[];slot_ids=[];series_uids=[];sources=[];triplets=[];primary=[];intensity=[]\n    for slot,(entry,k) in enumerate(zip(selected,allocation)):\n        name,_,_,_,reference_count=SLOTS[slot]\n        if entry is None:\n            selection.append(dict(slot=name,present=False,triplets=0));continue\n        series,files,refs,lo,hi=entry\n        details=dict(slot=name,present=True,series_uid=series[\'uid\'],description=series[\'description\'],\n            weight=series[\'weight\'],native_depth=len(files),eligible_lo=lo,eligible_hi=hi,\n            eligible_count=hi-lo+1,reference_allocation=reference_count,reference_count=len(refs),triplets=int(k))\n        if not k:selection.append({**details,\'sampled_count\':0});continue\n        picks,windows,disjoint=series_windows(lo,hi,int(k))\n        arrays={int(i):parent.pixels(files[int(i)][\'file\']) for i in sorted(set(refs)|set(picks))}\n        low,high=np.percentile(np.concatenate([arrays[int(i)].ravel() for i in refs]),[2,98])\n        offset=len(slices)\n        for native_index in picks:\n            record=files[int(native_index)]\n            slices.append(parent.crop_plane(arrays[int(native_index)],record[\'spacing\'],low,high))\n            slot_ids.append(slot);series_uids.append(series[\'uid\'])\n            sources.append(dict(slot=slot,series_uid=series[\'uid\'],file=Path(record[\'file\']).name,\n                sop_uid=record[\'sop_uid\'],native_index=int(native_index),position=record[\'position\'],spacing=record[\'spacing\']))\n        triplets.extend((windows+offset).tolist());primary.extend(disjoint.tolist())\n        selection.append({**details,\'sampled_count\':len(picks),\'disjoint_subset\':int(disjoint.sum())})\n        intensity.append(dict(slot=slot,p2=float(low),p98=float(high),reference_indices=refs.tolist()))\n    slices=np.stack(slices);indices=np.asarray(triplets,np.int32);slot_ids=np.asarray(slot_ids,np.int8)\n    series_uids=np.asarray(series_uids);usage=np.bincount(indices.ravel(),minlength=len(slices)).astype(np.int8)\n    audit=dict(study_uid=uid,recipe_sha256=RECIPE_SHA,parent_recipe_sha256=PARENT_SHA,\n        selection=selection,intensity=intensity,source_files=sources,source_slices=len(slices),windows=len(indices),\n        cross_series_windows=0,eligible_capacities=capacities,triplet_allocation=allocation.tolist(),\n        repeated_slice_uses=int(3*len(indices)-len(slices)),reused_source_slices=int((usage>1).sum()),\n        max_slice_usage=int(usage.max()),disjoint_subset_size=int(sum(primary)),**optimal)\n    data=dict(slices_uint8=slices,slice_slot_ids=slot_ids,slice_series_uids=series_uids,\n        triplet_indices=indices,triplet_series_uids=series_uids[indices[:,0]],\n        images=parent.normalized_triplets(slices,indices),triplet_slot_ids=slot_ids[indices],\n        same_series=np.ones(len(indices),bool),slot_present=np.asarray([s[\'present\'] for s in selection],bool),\n        slice_usage_counts=usage,primary_disjoint_subset=np.asarray(primary,bool),audit_json=np.asarray(json.dumps(audit,allow_nan=False)))\n    validate_arrays(data)\n    return data,audit\n\n\ndef validate_arrays(data):\n    BASE_VALIDATE(data)\n    indices=data[\'triplet_indices\'];audit=json.loads(str(data[\'audit_json\']))\n    if len(indices)>60 or not np.all(np.diff(indices,axis=1)==1):raise ValueError(\'Invalid neighboring triplet\')\n    if len({tuple(row) for row in indices})!=len(indices):raise ValueError(\'Duplicate triplet\')\n    usage=np.bincount(indices.ravel(),minlength=len(data[\'slices_uint8\']))\n    if not (usage>0).all() or not np.array_equal(usage,data[\'slice_usage_counts\']):raise ValueError(\'Invalid source reuse counts\')\n    if len({r[\'sop_uid\'] for r in audit[\'source_files\']})!=len(usage):raise ValueError(\'Reused or duplicate source SOP\')\n    allocation,optimal=allocate_triplets(audit[\'eligible_capacities\'])\n    if audit[\'recipe_sha256\']!=RECIPE_SHA or audit[\'triplet_allocation\']!=allocation.tolist():raise ValueError(\'Wrong sampling recipe\')\n    if len(indices)!=optimal[\'target_triplets\'] or len(usage)!=optimal[\'optimal_unique_slices\']:\n        raise ValueError(\'Non-optimal unique-slice coverage\')\n    if 3*len(indices)-len(usage)!=optimal[\'minimum_repeated_slice_uses\']:raise ValueError(\'Avoidable slice reuse\')\n    primary=data[\'primary_disjoint_subset\']\n    if primary.shape!=(len(indices),) or primary.sum()!=optimal[\'optimal_disjoint_subset\']:\n        raise ValueError(\'Non-optimal disjoint subset\')\n    members=indices[primary].ravel()\n    if len(np.unique(members))!=len(members):raise ValueError(\'Primary subset overlaps\')\n\n\ndef run(shard=0,shards=32,limit=None,root=None,output=None,workers=4):\n    # Reuse the exact label, Gold74, save/reopen/checksum and progress machinery.\n    # Only this notebook process\'s imported parent module is configured.\n    parent.prepare=prepare;parent.validate_arrays=validate_arrays\n    parent.RECIPE=RECIPE;parent.RECIPE_SHA=RECIPE_SHA\n    return parent.run(shard=shard,shards=shards,limit=limit,root=root,\n        output=output or f\'/kaggle/working/min_overlap_384_shard_{shard:02d}\',workers=workers)\n', 'infer_cv5_submission.py': '"""Five-fold EMA Raptor v5 + ResNet50 fusion Kaggle submission.\n\nInternet off. For every study in sample_submission.csv:\n  DICOM -> minimum-overlap 384 preprocessing (the exact V2 training recipe)\n  -> all same-series windows -> five fused models, float32, full bag\n  -> average sigmoid probabilities -> test.csv.\nNo test-time augmentation or Gold74-based checkpoint selection.\n"""\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom collections import deque\nfrom concurrent.futures import ProcessPoolExecutor\nfrom multiprocessing import get_context\nfrom pathlib import Path\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",\n          "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nTIMM_SHA = \'888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0\'\nCHECKPOINT_NAME = \'fold_{fold}_ema.pt\'\nEVAL_CHUNK = 4\nINPUT = Path(os.environ.get(\'RSNA_INPUT\', \'/kaggle/input\'))\n\n\ndef log(kind, **data):\n    print(json.dumps(dict(event=kind, utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()), **data)), flush=True)\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(8*1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef attached_roots():\n    """Attached datasets/notebook outputs, never the competition DICOM tree (too many files to walk)."""\n    roots = []\n    for base in (INPUT/\'datasets\', INPUT/\'notebooks\'):\n        if base.is_dir():\n            roots += [p for owner in base.iterdir() if owner.is_dir() for p in owner.iterdir() if p.is_dir()]\n    roots += [p for p in INPUT.iterdir() if p.is_dir() and p.name not in\n              (\'datasets\', \'notebooks\', \'competitions\', \'models\', \'rsna-knee-abnormality-detection\')]\n    return roots\n\n\ndef find_one(name, slug):\n    hits = sorted({p for r in attached_roots() if slug in str(r) for p in r.rglob(name)})\n    if len(hits) != 1:\n        raise FileNotFoundError(f\'Expected exactly one {name} under {slug}; found {hits}\')\n    return hits[0]\n\n\ndef ensure_timm():\n    try:\n        version = importlib.metadata.version(\'timm\')\n    except importlib.metadata.PackageNotFoundError:\n        version = None\n    if version != \'1.0.22\' and not os.environ.get(\'RSNA_SKIP_TIMM_INSTALL\'):\n        wheel = find_one(\'timm-1.0.22-py3-none-any.whl\', \'rsna-knee-coatnet-d4-depthzone-swa3-b2\')\n        if sha256(wheel) != TIMM_SHA:\n            raise RuntimeError(\'timm wheel hash mismatch\')\n        subprocess.check_call([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'--no-deps\', \'--no-index\', str(wheel)])\n        importlib.invalidate_caches()\n    for mod in [m for m in sys.modules if m == \'timm\' or m.startswith(\'timm.\')]:\n        del sys.modules[mod]\n    import timm\n    if timm.__version__ != \'1.0.22\' and not os.environ.get(\'RSNA_SKIP_TIMM_INSTALL\'):\n        raise RuntimeError(f\'timm {timm.__version__} loaded; 1.0.22 required\')\n    return timm.__version__\n\n\ndef competition_root():\n    for p in (INPUT, INPUT/\'competitions\'/\'rsna-knee-abnormality-detection\', INPUT/\'rsna-knee-abnormality-detection\'):\n        if (p/\'sample_submission.csv\').is_file() and (p/\'test_series.csv\').is_file():\n            return p\n    raise FileNotFoundError(\'Attach the rsna-knee-abnormality-detection competition data\')\n\n\n# ----------------------------------------------------------------------------- preprocessing workers\n_GROUPS, _SERIES_ROOT = None, None\n\n\ndef _init_worker(groups, series_root):\n    global _GROUPS, _SERIES_ROOT\n    import cv2\n    import torch\n    cv2.setNumThreads(1)\n    torch.set_num_threads(1)\n    _GROUPS, _SERIES_ROOT = groups, Path(series_root)\n\n\ndef _prepare(uid):\n    """Returns (uid, float16 [K,3,384,384] same-series windows, n_all_windows, error)."""\n    import numpy as np\n    from preprocess_min_overlap_384 import prepare\n    started = time.time()\n    try:\n        rows = _GROUPS.get(uid)\n        if not rows:\n            raise ValueError(\'No test_series rows for study\')\n        data, audit = prepare(uid, rows, _SERIES_ROOT)\n        images, same = data[\'images\'], np.asarray(data[\'same_series\'], bool)\n        if not same.all():\n            raise ValueError(\'Cross-series test window\')\n        if images.dtype != np.float16 or images.ndim != 4 or images.shape[1:] != (3, 384, 384) or not len(images):\n            raise ValueError(f\'Bad prepared tensor {images.shape} {images.dtype}\')\n        return uid, np.ascontiguousarray(images), int(audit[\'windows\']), None, round(time.time()-started, 2)\n    except Exception as error:   # noqa: BLE001 - recorded and handled by the caller\n        return uid, None, 0, f\'{type(error).__name__}: {str(error)[:300]}\', round(time.time()-started, 2)\n\n\n# ----------------------------------------------------------------------------- main\ndef main():\n    t0 = time.time()\n    timm_version = ensure_timm()\n    import numpy as np\n    import pandas as pd\n    import torch\n    from models.raptor_v5_resnet50 import load_fused_checkpoint\n\n    root = competition_root()\n    sample = pd.read_csv(root/\'sample_submission.csv\', dtype={\'StudyInstanceUID\': str})\n    if list(sample.columns) != [\'StudyInstanceUID\', *LABELS]:\n        raise ValueError(f\'Unexpected submission columns: {list(sample.columns)}\')\n    ids = list(sample.StudyInstanceUID)\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Duplicate study IDs in sample_submission.csv\')\n    series = pd.read_csv(root/\'test_series.csv\', dtype={\'StudyInstanceUID\': str, \'SeriesInstanceUID\': str})\n    groups = {uid: g.to_dict(\'records\') for uid, g in series.groupby(\'StudyInstanceUID\', sort=False)}\n    series_root = next((root/p for p in (\'test_series\', \'test_images\', \'test\') if (root/p).is_dir()), None)\n    if series_root is None:\n        raise FileNotFoundError(\'No test DICOM directory\')\n    workers = min(3, max(1, (os.cpu_count() or 2) - 1))\n    log(\'inputs\', studies=len(ids), studies_with_series=sum(u in groups for u in ids), series_root=str(series_root),\n        workers=workers, timm=timm_version, torch=torch.__version__, gpus=torch.cuda.device_count())\n\n    # Spawn allows a failed worker to be restarted safely after CUDA is initialised.\n    def new_pool():\n        return ProcessPoolExecutor(max_workers=workers, mp_context=get_context(\'spawn\'),\n                                   initializer=_init_worker, initargs=(groups, str(series_root)))\n\n    pool = new_pool()\n\n    ck_dir = (Path(os.environ[\'RSNA_CHECKPOINT_DIR\']) if \'RSNA_CHECKPOINT_DIR\' in os.environ\n              else find_one(\'manifest.json\', \'rsna-raptor-resnet50-cv5-ema\').parent)\n    manifest_path = ck_dir/\'manifest.json\'\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f\'Missing checkpoint manifest: {manifest_path}\')\n    manifest = json.loads(manifest_path.read_text())\n    from preprocess_min_overlap_384 import RECIPE_SHA\n    cuda = torch.cuda.is_available()\n    device = torch.device(\'cuda\', 0) if cuda else torch.device(\'cpu\')\n    # The Kaggle T4 image cannot execute the fused model\'s fp16 depthwise\n    # convolution through cuDNN; full-float32 inference avoids that kernel.\n    amp = dict(device_type=\'cuda\' if cuda else \'cpu\', enabled=False)\n    torch.backends.cudnn.benchmark = False\n    models, checkpoint_hashes, provenances = [], [], []\n    expected_epochs = [3, 3, 4, 3, 1]\n    if manifest[\'recipe_sha256\'] != RECIPE_SHA or len(manifest[\'checkpoints\']) != 5:\n        raise ValueError(\'Checkpoint manifest has an unexpected recipe or fold count\')\n    for fold in range(5):\n        entry = manifest[\'checkpoints\'][fold]\n        ck_path = ck_dir/CHECKPOINT_NAME.format(fold=fold)\n        checkpoint_sha256 = sha256(ck_path)\n        if (entry[\'fold\'] != fold or entry[\'file\'] != ck_path.name\n                or entry[\'bytes\'] != ck_path.stat().st_size or entry[\'sha256\'] != checkpoint_sha256):\n            raise ValueError(f\'Fold {fold} checkpoint integrity mismatch\')\n        ck = torch.load(ck_path, map_location=\'cpu\', weights_only=True)\n        provenance = dict(fold=ck[\'fold\'], epoch=ck[\'epoch\'], weight_kind=ck[\'weight_kind\'],\n                          recipe_sha256=ck[\'training_config\'][\'recipe_sha256\'])\n        if (provenance[\'fold\'] != fold or provenance[\'epoch\'] != expected_epochs[fold]\n                or provenance[\'weight_kind\'] != \'ema\' or provenance[\'recipe_sha256\'] != RECIPE_SHA):\n            raise ValueError(f\'Wrong fold {fold} model provenance: {provenance}\')\n        model = load_fused_checkpoint(ck).to(device).eval()\n        models.append(model)\n        checkpoint_hashes.append(checkpoint_sha256)\n        provenances.append(provenance)\n        del ck, model\n        log(\'model_loaded\', fold=fold, checkpoint=str(ck_path), sha256=checkpoint_sha256,\n            provenance=provenance)\n    log(\'ensemble_loaded\', folds=len(models), parameters_each=sum(p.numel() for p in models[0].parameters()))\n\n    preds, failures, windows, prep_seconds = {}, {}, {}, []\n    done = 0\n    # Bounded look-ahead: at most `ahead` prepared studies (~55 MB each) wait in memory.\n    queue, remaining, ahead = deque(), iter(ids), 2*workers + 2\n\n    def submit_next():\n        uid = next(remaining, None)\n        if uid is not None:\n            queue.append((uid, pool.submit(_prepare, uid)))\n\n    for _ in range(ahead):\n        submit_next()\n    with torch.inference_mode():\n        while queue:\n            uid, future = queue.popleft()\n            try:\n                uid, images, n_all, error, seconds = future.result()\n            except Exception as error:  # A worker can exit while decoding a damaged DICOM.\n                failures[uid] = f\'worker {type(error).__name__}: {str(error)[:300]}\'\n                log(\'study_failed\', study=uid, error=failures[uid])\n                pending = [study for study, _ in queue]\n                queue.clear()\n                pool.shutdown(wait=False, cancel_futures=True)\n                pool = new_pool()\n                queue.extend((study, pool.submit(_prepare, study)) for study in pending)\n                submit_next()\n                done += 1\n                continue\n            submit_next()\n            done += 1\n            prep_seconds.append(seconds)\n            if error is not None:\n                failures[uid] = error\n                log(\'study_failed\', study=uid, error=error)\n                continue\n            try:\n                x = torch.from_numpy(images)\n                n = len(x)\n                probabilities = []\n                for model in models:\n                    feats = []\n                    for chunk in x.split(EVAL_CHUNK):\n                        with torch.autocast(**amp):\n                            feats.append(model.backbone(chunk.to(device, non_blocking=True).float()))\n                    with torch.autocast(**amp):\n                        logits = model.head(torch.cat(feats, 0)[:n][None])\n                    probabilities.append(torch.sigmoid(logits.float())[0].cpu().numpy())\n                p = np.mean(probabilities, axis=0)\n                if not np.isfinite(p).all():\n                    raise ValueError(\'non-finite prediction\')\n            except Exception as error:  # A single unusual test study must not abort the submission.\n                failures[uid] = f\'{type(error).__name__}: {str(error)[:300]}\'\n                log(\'study_failed\', study=uid, error=failures[uid])\n                if cuda:\n                    torch.cuda.empty_cache()\n                continue\n            preds[uid] = p\n            windows[uid] = (n, n_all)\n            if done <= 3 or done % 100 == 0 or done == len(ids):\n                log(\'progress\', done=done, total=len(ids), failed=len(failures), windows_used=n, windows_all=n_all,\n                    elapsed_min=round((time.time()-t0)/60, 2))\n    pool.shutdown()\n\n    # Kaggle\'s hidden DICOMs can include an unreadable or otherwise unusual study.\n    # Give those rows the per-label mean of successfully predicted studies, as in\n    # the fusion notebook that already completed a scored hidden-data rerun.\n    # Keep every failure in the receipt so fallback use is visible and auditable.\n    fallback = (np.mean(np.stack(list(preds.values())), axis=0)\n                if preds else np.full(len(LABELS), 0.5, dtype=np.float32))\n    if failures:\n        log(\'fallback_predictions\', failed=len(failures), predicted=len(preds),\n            fallback=\'mean_predicted_probabilities\' if preds else \'neutral_0.5\')\n    rows = [[uid, *preds.get(uid, fallback).tolist()] for uid in ids]\n    sub = pd.DataFrame(rows, columns=[\'StudyInstanceUID\', *LABELS])\n    values = sub[LABELS].to_numpy(float)\n    if sub.shape != sample.shape or not np.isfinite(values).all() or ((values < 0) | (values > 1)).any():\n        raise RuntimeError(\'Invalid submission\')\n    sub.to_csv(\'test.csv\', index=False)\n    # This competition requires the notebook output file to use this exact name.\n    Path(\'submission.csv\').write_bytes(Path(\'test.csv\').read_bytes())\n    receipt = dict(model=\'Raptor v5 + ResNet50 fusion (five-fold EMA probability ensemble, no TTA)\',\n                   checkpoint_dir=str(ck_dir), checkpoint_sha256=checkpoint_hashes,\n                   provenance=provenances, recipe_sha256=RECIPE_SHA,\n                   studies=len(ids), predicted=len(preds), fallback_count=len(failures),\n                   failed=failures, mean_windows_used=(float(np.mean([w[0] for w in windows.values()]))\n                                                      if windows else 0.0),\n                   prep_seconds_median=float(np.median(prep_seconds)), minutes=round((time.time()-t0)/60, 2))\n    Path(\'inference_receipt.json\').write_text(json.dumps(receipt, indent=2))\n    log(\'submission_written\', **{k: v for k, v in receipt.items() if k != \'failed\'}, failed=len(failures))\n\n\nif __name__ == \'__main__\':\n    main()\n'}
src = Path('/kaggle/working/fusion_src')
for name, text in SOURCES.items():
    path = src/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
print('sources written:', sorted(SOURCES), flush=True)


In [ ]:
import os, subprocess, sys
env = dict(os.environ, PYTHONUNBUFFERED='1', PYTHONPATH=str(src), OMP_NUM_THREADS='1')
proc = subprocess.Popen([sys.executable, str(src/'infer_cv5_submission.py')], stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, env=env, cwd='/kaggle/working', bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
if proc.wait() != 0:
    raise SystemExit(f'Inference failed with exit code {proc.returncode}')
import pandas as pd
sub = pd.read_csv('/kaggle/working/test.csv')
print(sub.shape)
print(sub.head())
